# API 계약에서 컨테이너와 저장소까지

조회 API가 성공을 반환해도 다른 대화의 문서가 섞일 수 있다. 요청·저장·응답의 식별자를 끝까지 추적한다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 패키지가 없는 커널에서는 새 코드 셀에 다음 명령을 실행한 뒤 예제로 이동하세요.

%pip install "httpx>=0.27,<1"

## HTTP 상태와 업무 결과 읽기

상담 조회 API에는 존재하는 접수, 없는 접수, 형식이 잘못된 입력이 들어온다. 세 경우에 모두 빈 배열을 반환하면 클라이언트가 무엇을 고쳐야 할지 판단할 수 없다.

화면이 보내는 사용자·대화 ID, 서버가 찾은 레코드, DB의 복합 키를 같은 표에서 대조한다. 이후 컨테이너와 DB를 분리하더라도 이 계약을 테스트 입력으로 재사용할 수 있다.

아래에서는 HTTP 요청·응답을 가상 전송 계층으로 확인한다. 서버·Compose 시작, 볼륨 재개방, pgvector 검색은 개별 실습의 실제 프로세스에서 시험한다.

## 같은 접수 ID를 다른 사용자가 보냈을 때

httpx의 MockTransport로 응답 분기를 관찰한다. 예제의 사용자 값은 라우팅 시험용이며 서비스의 인증을 구현한 값은 아니다.

In [1]:
import httpx
records = {("u1","T4"): "검토 중"}
def service(req):
    key = (req.headers.get("x-demo-user"), req.url.path.rsplit("/",1)[-1])
    if key not in records:
        return httpx.Response(404, json={"detail":"찾을 수 없음"})
    return httpx.Response(200, json={"status":records[key]})

client = httpx.Client(base_url="http://classroom.test", transport=httpx.MockTransport(service))
print(client.get("/tickets/T4", headers={"x-demo-user":"u1"}).json())

{'status': '검토 중'}


사용자와 접수 ID가 함께 맞아야 레코드를 읽는다. 성공 응답을 한 번 보는 것만으로는 사용자 격리가 작동하는지 판단할 수 없다.

### 응답 본문에서 다른 사람의 상태가 새지 않는지 확인

같은 경로를 u2로 호출하고 상태 코드와 본문을 같이 확인한다.

In [2]:
path = "/tickets/T4"

In [3]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>missing = client.get(path, headers={&quot;x-demo-user&quot;:&quot;u2&quot;})
print(missing.status_code, missing.json())</code></pre><pre>404 {&#x27;detail&#x27;: &#x27;찾을 수 없음&#x27;}
</pre><p>404 응답에는 T4의 상태가 없다. 공개 오류 문구에 원본 고객 정보가 들어가지는 않았는지도 확인한다.</p></details>

## 브라우저에 잘린 스트림이 도착하기

서버 응답 한 줄이 네트워크 조각 하나와 일치한다는 보장은 없다. 줄 경계를 복구한 뒤에만 JSON을 읽어 보자.

In [4]:
import json
pieces = ['{"seq":1,"text":"안', '내"}\n{"seq":2,', '"text":" 끝"}\n']
def read_lines(pieces):
    tail = ""
    for piece in pieces:
        tail += piece
        while "\n" in tail:
            line, tail = tail.split("\n", 1)
            if line:
                yield json.loads(line)
    if tail:
        raise ValueError("완료되지 않은 줄")
print(list(read_lines(pieces)))

[{'seq': 1, 'text': '안내'}, {'seq': 2, 'text': ' 끝'}]


첫 번째 조각에는 닫는 따옴표도 없다. 이 예제의 전송 형식은 한 줄에 JSON 한 건이며, SSE를 쓰면 event와 data 줄의 경계를 그 형식대로 처리해야 한다.

### 연결이 중간에 끊겼을 때 완성된 줄만 보존하기

마지막 조각을 빼고 읽는다. 첫 줄은 이미 도착했고 두 번째 줄은 미완성이다. 두 상태를 사용자에게 구별해 전달하도록 기록한다.

In [5]:
complete = []

In [6]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>try:
    for record in read_lines(pieces[:-1]):
        complete.append(record[&quot;seq&quot;])
except ValueError as error:
    print(complete, str(error))</code></pre><pre>[1] 완료되지 않은 줄
</pre><p>완료한 seq 1만 남는다. 재연결할 때 같은 번호가 다시 도착하면 중복 게시하지 않는 조건도 이어서 시험할 수 있다.</p></details>

## 다른 입력에 적용하기

동일한 두 요청을 FastAPI 앱과 Compose로 띄운 API에 보낸다. DB 컨테이너를 다시 띄운 뒤 같은 결과가 나오는지 비교하고, 검색을 추가하면 문서 ID와 소유자 조건도 함께 점검한다.

로컬에서 통과한 HTTP 시험, 실제 DB와 연결한 시험, 컨테이너 재시작 뒤의 관찰을 분리해 기록한다.